## Feature Engineering - Customer Churn

### By:
Author Name

### Date:
2026-02-23

### Description:

Create a feature engineering pipeline based on the data analysis of the customer churn dataset. 
The feature engineering will be performed using scikit-learn pipelines and transformers to prepare the data for machine learning models.


## 📚 Import libraries

In [1]:
# base libraries for data science
from pathlib import Path

import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

## 💾 Load data

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

churn_df = pd.read_parquet(DATA_DIR / "02_intermediate/churn_cleaned.parquet", engine="pyarrow")

In [3]:
# print library versions for reproducibility
print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

Pandas version:  2.3.3
sklearn version:  1.8.0


In [4]:
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14214 entries, 0 to 14213
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   SeniorCitizen     14154 non-null  float64 
 1   Partner           14153 non-null  category
 2   Dependents        14124 non-null  category
 3   tenure            14114 non-null  float64 
 4   MultipleLines     14077 non-null  category
 5   InternetService   14053 non-null  category
 6   OnlineSecurity    14024 non-null  category
 7   OnlineBackup      14013 non-null  category
 8   DeviceProtection  14020 non-null  category
 9   TechSupport       14017 non-null  category
 10  StreamingTV       14015 non-null  category
 11  StreamingMovies   14036 non-null  category
 12  Contract          14092 non-null  category
 13  PaperlessBilling  14092 non-null  category
 14  PaymentMethod     14088 non-null  category
 15  MonthlyCharges    14101 non-null  float32 
 16  TotalCharges      1412

In [5]:
churn_df.head()

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0.0,Yes,No,1.0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.850000,29.85,No
1,0.0,No,No,34.0,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.950001,1889.50,No
2,0.0,No,No,2.0,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.849998,108.15,Yes
3,0.0,No,No,45.0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.299999,1840.75,No
4,0.0,No,No,2.0,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.699997,151.65,Yes


## 👷 Data preparation

Select the relevant features for the model and inspect data quality.

In [6]:
selected_features = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
]

churn_features = churn_df[selected_features].copy()
churn_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14214 entries, 0 to 14213
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   SeniorCitizen     14154 non-null  float64 
 1   Partner           14153 non-null  category
 2   Dependents        14124 non-null  category
 3   tenure            14114 non-null  float64 
 4   MultipleLines     14077 non-null  category
 5   InternetService   14053 non-null  category
 6   OnlineSecurity    14024 non-null  category
 7   OnlineBackup      14013 non-null  category
 8   DeviceProtection  14020 non-null  category
 9   TechSupport       14017 non-null  category
 10  StreamingTV       14015 non-null  category
 11  StreamingMovies   14036 non-null  category
 12  Contract          14092 non-null  category
 13  PaperlessBilling  14092 non-null  category
 14  PaymentMethod     14088 non-null  category
 15  MonthlyCharges    14101 non-null  float32 
 16  TotalCharges      1412

### Missing values

In [7]:
# Replace blank strings ' ' with NaN — only on object/string columns to avoid
# crashing on category dtype columns loaded from Parquet.
string_cols = churn_features.select_dtypes(include=["object"]).columns
churn_features[string_cols] = churn_features[string_cols].replace(
    r"^\s*$", float("nan"), regex=True
)

# TotalCharges may still be string/category if loaded from CSV path — coerce to numeric
churn_features["TotalCharges"] = pd.to_numeric(churn_features["TotalCharges"], errors="coerce")

churn_features.isna().sum()

SeniorCitizen        60
Partner              61
Dependents           90
tenure              100
MultipleLines       137
InternetService     161
OnlineSecurity      190
OnlineBackup        201
DeviceProtection    194
TechSupport         197
StreamingTV         199
StreamingMovies     178
Contract            122
PaperlessBilling    122
PaymentMethod       126
MonthlyCharges      113
TotalCharges         93
Churn                89
dtype: int64

### Duplicated data

In [8]:
duplicate_rows = churn_features.duplicated().sum()
print("Number of duplicate rows: ", duplicate_rows)

Number of duplicate rows:  6979


In [9]:
churn_features = churn_features.drop_duplicates()
churn_features.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7235 entries, 0 to 14149
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   SeniorCitizen     7199 non-null   float64 
 1   Partner           7199 non-null   category
 2   Dependents        7170 non-null   category
 3   tenure            7159 non-null   float64 
 4   MultipleLines     7123 non-null   category
 5   InternetService   7100 non-null   category
 6   OnlineSecurity    7069 non-null   category
 7   OnlineBackup      7063 non-null   category
 8   DeviceProtection  7067 non-null   category
 9   TechSupport       7065 non-null   category
 10  StreamingTV       7064 non-null   category
 11  StreamingMovies   7081 non-null   category
 12  Contract          7140 non-null   category
 13  PaperlessBilling  7139 non-null   category
 14  PaymentMethod     7137 non-null   category
 15  MonthlyCharges    7147 non-null   float32 
 16  TotalCharges      7177 non-n

## 👨‍🏭 Feature Engineering

Build a scikit-learn preprocessing pipeline to handle imputation, encoding, and scaling for each type of feature.

In [10]:
# Encode target variable: Yes -> 1, No -> 0
churn_features["Churn"] = (churn_features["Churn"] == "Yes").astype(int)

churn_features["Churn"].value_counts()

Churn
0    5336
1    1899
Name: count, dtype: int64

In [11]:
churn_features.sample(5, random_state=42)

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5805,1.0,Yes,No,19.0,No,DSL,Yes,No,Yes,No,Yes,No,Month-to-month,No,Bank transfer (automatic),66.400002,1286.05,0
743,0.0,Yes,Yes,61.0,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,24.100000,1551.60,0
2504,0.0,No,No,2.0,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,75.699997,189.20,1
6370,0.0,No,No,45.0,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.300003,4016.85,1
3964,0.0,Yes,Yes,68.0,Yes,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),111.750000,7511.30,0


In [12]:
# Numeric features - continuous
cols_numeric = ["tenure", "MonthlyCharges", "TotalCharges"]

# Binary categorical features (Yes/No)
cols_categoric_binary = [
    "Partner",
    "Dependents",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling",
]

# Multi-class categorical features
cols_categoric_nominal = ["InternetService", "PaymentMethod"]

# Ordinal categorical features
cols_categoric_ordinal = ["Contract"]  # Month-to-month < One year < Two year

# Integer / binary numeric feature
cols_numeric_binary = ["SeniorCitizen"]  # already encoded as 0/1

In [ ]:
# Numeric pipeline: impute with median + scale
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Binary categorical pipeline: impute + one-hot encode (drop='if_binary' keeps 1 column)
categorical_binary_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="if_binary", sparse_output=False, handle_unknown="ignore")),
    ]
)

# Nominal categorical pipeline: impute + one-hot encode
categorical_nominal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]
)

# Ordinal categorical pipeline: impute + ordinal encode
contract_order = [["Month-to-month", "One year", "Two year"]]
categorical_ordinal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=contract_order, handle_unknown="use_encoded_value", unknown_value=-1
            ),
        ),
    ]
)

# Passthrough for already-encoded binary numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("categoric_binary", categorical_binary_pipe, cols_categoric_binary),
        ("categoric_nominal", categorical_nominal_pipe, cols_categoric_nominal),
        ("categoric_ordinal", categorical_ordinal_pipe, cols_categoric_ordinal),
        ("passthrough", "passthrough", cols_numeric_binary),
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categoric_binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``


**Pipeline description:**

1. **Numeric Pipeline** — `["tenure", "MonthlyCharges", "TotalCharges"]`:
   - `SimpleImputer(strategy="median")`: fills missing values with the median.
   - `StandardScaler()`: standardizes features to zero mean and unit variance.

2. **Binary Categorical Pipeline** — e.g. `Partner`, `Dependents`, `OnlineSecurity`, …:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OneHotEncoder(drop="if_binary", handle_unknown="ignore")`: encodes Yes/No as a single 0/1 column; unknown categories at inference time are silently encoded as 0.

3. **Nominal Categorical Pipeline** — `["InternetService", "PaymentMethod"]`:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OneHotEncoder(handle_unknown="ignore")`: creates one binary column per category; unknown categories at inference time are encoded as all-zeros.

4. **Ordinal Categorical Pipeline** — `["Contract"]`:
   - `SimpleImputer(strategy="most_frequent")`: fills missing values with the mode.
   - `OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]], handle_unknown="use_encoded_value", unknown_value=-1)`: encodes with meaningful order (0 < 1 < 2); unknown values are mapped to -1.

5. **Passthrough** — `["SeniorCitizen"]`: already encoded as 0/1, passed through unchanged.


## Example of the data preprocessing pipeline

### Train / Test split

In [14]:
X_features = churn_features.drop("Churn", axis="columns")
Y_target = churn_features["Churn"]

# 80% train, 20% test — stratify to preserve churn ratio
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, random_state=42, stratify=Y_target
)

print(f"Train set: {x_train.shape}, Test set: {x_test.shape}")

Train set: (5788, 17), Test set: (1447, 17)


### Preprocessing pipeline

In [15]:
# Fit preprocessor on training data only to avoid data leakage
preprocessor.fit(x_train)

feature_names = preprocessor.get_feature_names_out()

x_train_transformed = pd.DataFrame(preprocessor.transform(x_train), columns=feature_names)
x_test_transformed = pd.DataFrame(preprocessor.transform(x_test), columns=feature_names)

x_train_transformed.info()

ValueError: Found unknown categories ['1523434'] in column 8 during transform

In [16]:
x_train_transformed.head()

,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,categoric_binary__Partner_Yes,categoric_binary__Dependents_Yes,categoric_binary__MultipleLines_1244132,categoric_binary__MultipleLines_No,categoric_binary__MultipleLines_No phone service,categoric_binary__MultipleLines_Yes,categoric_binary__OnlineSecurity_23453432,...,categoric_binary__PaperlessBilling_Yes,categoric_nominal__InternetService_DSL,categoric_nominal__InternetService_Fiber optic,categoric_nominal__InternetService_No,categoric_nominal__PaymentMethod_Bank transfer (automatic),categoric_nominal__PaymentMethod_Credit card (automatic),categoric_nominal__PaymentMethod_Electronic check,categoric_nominal__PaymentMethod_Mailed check,categoric_ordinal__Contract,passthrough__SeniorCitizen
0,0.020420,-0.013145,-0.013119,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0
1,-1.127877,-0.013145,-0.013119,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.758612,-0.013145,-0.013119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,-1.127877,-0.013145,-0.013119,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-0.102612,-0.013145,-0.013119,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
# Verify no missing values remain after transformation
print("Missing values in transformed train set:", x_train_transformed.isna().sum().sum())
print("Missing values in transformed test set: ", x_test_transformed.isna().sum().sum())

## 💾 Save Processed Data and Preprocessor

In [ ]:
import json

import joblib

# ── Directories ────────────────────────────────────────────────────────────────
model_input_dir = DATA_DIR / "05_model_input"
models_dir = DATA_DIR / "06_models"
model_input_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# ── Save train/test splits (features + target) ─────────────────────────────────
x_train_transformed.to_parquet(model_input_dir / "x_train.parquet", index=False)
x_test_transformed.to_parquet(model_input_dir / "x_test.parquet", index=False)
y_train.reset_index(drop=True).to_frame().to_parquet(
    model_input_dir / "y_train.parquet", index=False
)
y_test.reset_index(drop=True).to_frame().to_parquet(model_input_dir / "y_test.parquet", index=False)

# ── Save fitted preprocessor ───────────────────────────────────────────────────
joblib.dump(preprocessor, models_dir / "preprocessor.pkl")
# Save feature names for downstream use

with open(models_dir / "feature_names.json", "w") as f:
    json.dump(list(feature_names), f)

print("✅ Saved to data/05_model_input/:")
print(f"   x_train.parquet  → {x_train_transformed.shape}")
print(f"   x_test.parquet   → {x_test_transformed.shape}")
print(f"   y_train.parquet  → {y_train.shape}")
print(f"   y_test.parquet   → {y_test.shape}")
print("\n✅ Saved to data/06_models/:")
print("   preprocessor.pkl")
print(f"   feature_names.json ({len(feature_names)} features)")

## 📊 Analysis of Results and Conclusions

The feature engineering pipeline successfully transforms the raw churn dataset into a format ready for machine learning:

- **Numeric features** (`tenure`, `MonthlyCharges`, `TotalCharges`) are imputed and standardized, ensuring algorithms sensitive to scale (e.g., logistic regression, SVM) are not biased.
- **Binary Yes/No features** are efficiently encoded into single 0/1 columns, keeping dimensionality low.
- **Nominal features** (`InternetService`, `PaymentMethod`) are one-hot encoded to avoid introducing false ordinal relationships.
- **Ordinal feature** (`Contract`) is encoded respecting the natural order: Month-to-month (0) < One year (1) < Two year (2).
- **No data leakage**: the preprocessor is fitted only on training data and then applied to the test set.

The final transformed dataset has no missing values and all features are numeric, making it directly usable for classification models.


## 💡 Proposals and Ideas

1. **Advanced imputation**: Consider `KNNImputer` or `IterativeImputer` for `TotalCharges` if missing values are non-random (MNAR/MAR).

2. **New features via feature engineering**:
   - `services_count`: count of active services per customer (streaming, security, backup, etc.) to capture total engagement.
   - `charges_per_month_ratio`: `TotalCharges / tenure` as a proxy for pricing stability.
   - `is_new_customer`: binary flag for `tenure < 6` months, since new customers may have higher churn risk.

3. **Feature scaling alternatives**: Try `MinMaxScaler` or `RobustScaler` for numeric features if outliers are present in charges.

4. **Target encoding**: For high-cardinality categorical features (e.g., `PaymentMethod`), compare one-hot vs. target encoding impact on model performance.

5. **Class imbalance**: Explore SMOTE or class weighting in the model, since churn datasets are typically imbalanced (~26% churn in this dataset).


## 📖 References

- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/
- https://joserzapata.github.io/courses/python-ciencia-datos/ml/
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow — Aurélien Géron
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/preprocessing.html
